## Module 6: Basic NLP with NLTK
## NLP Analysis on Shakespeare's *Hamlet* from Project Gutenberg

**Student Name:** Dev Panchal
**Course:** EAI6000  

In this notebook, we will perform a complete NLP pipeline on Shakespeare's *Hamlet* using NLTK. The steps include:

1. Loading and exploring the corpus
2. Sentence and word tokenization
3. Lowercasing
4. Stopword removal
5. Stemming and Lemmatization
6. Vectorization (Bag-of-Words)
7. Building a Naive Bayes classifier to classify text by genre

---
## 1. Setup and Imports
Import libraries and download the NLTK data packages we need.

In [1]:
import nltk
import pandas as pd
import numpy as np
from collections import Counter
from string import punctuation

# Download all required NLTK data
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('brown')

print('All NLTK packages downloaded successfully!')

All NLTK packages downloaded successfully!


[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/devpanchal/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]

---
## 2. Loading the Corpus
NLTK has several texts from the Gutenberg Project built in. We will use Shakespeare's Hamlet.

In [2]:
from nltk.corpus import gutenberg

# See all available texts in the Gutenberg corpus
print('Available texts in NLTK Gutenberg corpus:')
print(gutenberg.fileids())

Available texts in NLTK Gutenberg corpus:
['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


In [3]:
# Load Hamlet as words, sentences, and raw text
hamlet_words = gutenberg.words('shakespeare-hamlet.txt')
hamlet_sents = gutenberg.sents('shakespeare-hamlet.txt')
hamlet_raw = gutenberg.raw('shakespeare-hamlet.txt')

# Show basic stats
print(f'Total words in Hamlet: {len(hamlet_words)}')
print(f'Total sentences in Hamlet: {len(hamlet_sents)}')
print(f'Total characters in Hamlet: {len(hamlet_raw)}')

Total words in Hamlet: 37360
Total sentences in Hamlet: 3106
Total characters in Hamlet: 162881


In [4]:
# Preview the first 500 characters of raw text
print('First 500 characters of Hamlet:')
print(hamlet_raw[:500])

First 500 characters of Hamlet:
[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not


In [5]:
# Preview first 5 sentences as word lists
print('First 5 sentences of Hamlet:')
for i, sent in enumerate(hamlet_sents[:5]):
    print(f'Sentence {i+1}: {sent}')

First 5 sentences of Hamlet:
Sentence 1: ['[', 'The', 'Tragedie', 'of', 'Hamlet', 'by', 'William', 'Shakespeare', '1599', ']']
Sentence 2: ['Actus', 'Primus', '.']
Sentence 3: ['Scoena', 'Prima', '.']
Sentence 4: ['Enter', 'Barnardo', 'and', 'Francisco', 'two', 'Centinels', '.']
Sentence 5: ['Barnardo', '.']


In [6]:
# Preview the first 30 words
print('First 30 words of Hamlet:')
print(hamlet_words[:30])

First 30 words of Hamlet:
['[', 'The', 'Tragedie', 'of', 'Hamlet', 'by', 'William', 'Shakespeare', '1599', ']', 'Actus', 'Primus', '.', 'Scoena', 'Prima', '.', 'Enter', 'Barnardo', 'and', 'Francisco', 'two', 'Centinels', '.', 'Barnardo', '.', 'Who', "'", 's', 'there', '?']


---
## 3. Tokenization
Tokenization means splitting text into smaller pieces. `sent_tokenize()` splits text into sentences, and `word_tokenize()` splits sentences into words.

In [7]:
from nltk import sent_tokenize, word_tokenize

# A famous passage from Hamlet for testing
sample_passage = "To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles."

# Split the passage into sentences
sentences = sent_tokenize(sample_passage)
print('Sentence Tokenization:')
for i, sent in enumerate(sentences):
    print(f'  Sentence {i+1}: {sent}')

Sentence Tokenization:
  Sentence 1: To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles.


In [8]:
# Split each sentence into individual words
print('Word Tokenization (per sentence):')
for sent in sentences:
    print(f'  {word_tokenize(sent)}')

Word Tokenization (per sentence):
  ['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'Whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'The', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'Or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


In [9]:
# Tokenize the entire passage at once
all_tokens = word_tokenize(sample_passage)
print(f'All tokens ({len(all_tokens)} total):')
print(all_tokens)

All tokens (40 total):
['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'Whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'The', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'Or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


---
## 4. Lowercasing
We convert all words to lowercase so that "The" and "the" are treated as the same word. We do this after tokenization because the tokenizer uses capitals to detect sentence boundaries.

In [10]:
# Tokenize first, then lowercase each word
print('Lowercased tokens:')
for sent in sent_tokenize(sample_passage):
    lowered = [word.lower() for word in word_tokenize(sent)]
    print(f'  {lowered}')

Lowercased tokens:
  ['to', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'the', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


In [11]:
# Lowercase all tokens from the full passage
all_tokens_lower = [word.lower() for word in word_tokenize(sample_passage)]
print(f'All lowercased tokens ({len(all_tokens_lower)}):')
print(all_tokens_lower)

All lowercased tokens (40):
['to', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'the', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']


---
## 5. Stopword Removal
Stopwords are common words like "the", "is", "and" that don't carry much meaning. Removing them helps us focus on the important words.

In [12]:
from nltk.corpus import stopwords

# Load English stopwords from NLTK
stopwords_en = set(stopwords.words('english'))
print(f'Total stopwords in NLTK: {len(stopwords_en)}')
print(f'Some examples: {list(stopwords_en)[:20]}')

Total stopwords in NLTK: 198
Some examples: ["we'll", 'itself', 'hers', 'too', 'only', 'o', "haven't", "we've", 'because', 'at', 'couldn', 'few', 'for', 'how', 'very', 'y', 'himself', 'no', 'then', 'ours']


In [13]:
# Add punctuation to the stopwords list
stopwords_en_withpunct = stopwords_en.union(set(punctuation))
print(f'Stopwords + punctuation: {len(stopwords_en_withpunct)} total')

Stopwords + punctuation: 230 total


In [14]:
# Use a bigger stopwords list from stopwords-json for better coverage
stopwords_json = {"en":["a","a's","able","about","above","according","accordingly","across","actually","after","afterwards","again","against","ain't","all","allow","allows","almost","alone","along","already","also","although","always","am","among","amongst","an","and","another","any","anybody","anyhow","anyone","anything","anyway","anyways","anywhere","apart","appear","appreciate","appropriate","are","aren't","around","as","aside","ask","asking","associated","at","available","away","awfully","b","be","became","because","become","becomes","becoming","been","before","beforehand","behind","being","believe","below","beside","besides","best","better","between","beyond","both","brief","but","by","c","c'mon","c's","came","can","can't","cannot","cant","cause","causes","certain","certainly","changes","clearly","co","com","come","comes","concerning","consequently","consider","considering","contain","containing","contains","corresponding","could","couldn't","course","currently","d","definitely","described","despite","did","didn't","different","do","does","doesn't","doing","don't","done","down","downwards","during","e","each","edu","eg","eight","either","else","elsewhere","enough","entirely","especially","et","etc","even","ever","every","everybody","everyone","everything","everywhere","ex","exactly","example","except","f","far","few","fifth","first","five","followed","following","follows","for","former","formerly","forth","four","from","further","furthermore","g","get","gets","getting","given","gives","go","goes","going","gone","got","gotten","greetings","h","had","hadn't","happens","hardly","has","hasn't","have","haven't","having","he","he's","hello","help","hence","her","here","here's","hereafter","hereby","herein","hereupon","hers","herself","hi","him","himself","his","hither","hopefully","how","howbeit","however","i","i'd","i'll","i'm","i've","ie","if","ignored","immediate","in","inasmuch","inc","indeed","indicate","indicated","indicates","inner","insofar","instead","into","inward","is","isn't","it","it'd","it'll","it's","its","itself","j","just","k","keep","keeps","kept","know","known","knows","l","last","lately","later","latter","latterly","least","less","lest","let","let's","like","liked","likely","little","look","looking","looks","ltd","m","mainly","many","may","maybe","me","mean","meanwhile","merely","might","more","moreover","most","mostly","much","must","my","myself","n","name","namely","nd","near","nearly","necessary","need","needs","neither","never","nevertheless","new","next","nine","no","nobody","non","none","noone","nor","normally","not","nothing","novel","now","nowhere","o","obviously","of","off","often","oh","ok","okay","old","on","once","one","ones","only","onto","or","other","others","otherwise","ought","our","ours","ourselves","out","outside","over","overall","own","p","particular","particularly","per","perhaps","placed","please","plus","possible","presumably","probably","provides","q","que","quite","qv","r","rather","rd","re","really","reasonably","regarding","regardless","regards","relatively","respectively","right","s","said","same","saw","say","saying","says","second","secondly","see","seeing","seem","seemed","seeming","seems","seen","self","selves","sensible","sent","serious","seriously","seven","several","shall","she","should","shouldn't","since","six","so","some","somebody","somehow","someone","something","sometime","sometimes","somewhat","somewhere","soon","sorry","specified","specify","specifying","still","sub","such","sup","sure","t","t's","take","taken","tell","tends","th","than","thank","thanks","thanx","that","that's","thats","the","their","theirs","them","themselves","then","thence","there","there's","thereafter","thereby","therefore","therein","theres","thereupon","these","they","they'd","they'll","they're","they've","think","third","this","thorough","thoroughly","those","though","three","through","throughout","thru","thus","to","together","too","took","toward","towards","tried","tries","truly","try","trying","twice","two","u","un","under","unfortunately","unless","unlikely","until","unto","up","upon","us","use","used","useful","uses","using","usually","uucp","v","value","various","very","via","viz","vs","w","want","wants","was","wasn't","way","we","we'd","we'll","we're","we've","welcome","well","went","were","weren't","what","what's","whatever","when","whence","whenever","where","where's","whereafter","whereas","whereby","wherein","whereupon","wherever","whether","which","while","whither","who","who's","whoever","whole","whom","whose","why","will","willing","wish","with","within","without","won't","wonder","would","wouldn't","x","y","yes","yet","you","you'd","you'll","you're","you've","your","yours","yourself","yourselves","z","zero"]}

# Combine all three stopword sources into one big list
stoplist_combined = set.union(set(stopwords_json['en']), set(stopwords.words('english')), set(punctuation))
print(f'Combined stopwords list size: {len(stoplist_combined)}')

Combined stopwords list size: 635


In [15]:
# Show before and after stopword removal
tokens_lower = [word.lower() for word in word_tokenize(sample_passage)]

print('Before stopword removal:')
print(tokens_lower)
print(f'Count: {len(tokens_lower)}\n')

# Keep only meaningful words
tokens_clean = [word for word in tokens_lower if word not in stoplist_combined]

print('After stopword removal:')
print(tokens_clean)
print(f'Count: {len(tokens_clean)}')

Before stopword removal:
['to', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question', ':', 'whether', "'t", 'is', 'nobler', 'in', 'the', 'mind', 'to', 'suffer', 'the', 'slings', 'and', 'arrows', 'of', 'outrageous', 'fortune', ',', 'or', 'to', 'take', 'arms', 'against', 'a', 'sea', 'of', 'troubles', '.']
Count: 40

After stopword removal:
['question', "'t", 'nobler', 'mind', 'suffer', 'slings', 'arrows', 'outrageous', 'fortune', 'arms', 'sea', 'troubles']
Count: 12


---
## 6. Stemming and Lemmatization
Both try to find the root form of a word. **Stemming** chops off endings using simple rules. **Lemmatization** uses grammar rules to find the proper dictionary root word.

In [16]:
from nltk.stem import PorterStemmer

# Porter Stemmer - uses rules to cut off word endings
porter = PorterStemmer()

test_words = ['walking', 'walks', 'walked', 'troubles', 'suffering', 'nobler']
print('Porter Stemmer Results:')
for word in test_words:
    print(f'  {word} -> {porter.stem(word)}')

Porter Stemmer Results:
  walking -> walk
  walks -> walk
  walked -> walk
  troubles -> troubl
  suffering -> suffer
  nobler -> nobler


In [17]:
from nltk.stem import WordNetLemmatizer

# WordNet Lemmatizer - without POS tags it assumes every word is a noun
wnl = WordNetLemmatizer()

print('Lemmatizer Results (without POS tags):')
for word in test_words:
    print(f'  {word} -> {wnl.lemmatize(word)}')

Lemmatizer Results (without POS tags):
  walking -> walking
  walks -> walk
  walked -> walked
  troubles -> trouble
  suffering -> suffering
  nobler -> nobler


In [18]:
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

wnl = WordNetLemmatizer()

def penn2morphy(penntag):
    """Convert Penn Treebank POS tags to WordNet format."""
    morphy_tag = {'NN': 'n', 'JJ': 'a', 'VB': 'v', 'RB': 'r'}
    try:
        return morphy_tag[penntag[:2]]
    except:
        return 'n'

def lemmatize_sent(text):
    """Tokenize, POS tag, and lemmatize a sentence."""
    return [wnl.lemmatize(word.lower(), pos=penn2morphy(tag))
            for word, tag in pos_tag(word_tokenize(text))]

# Test - now verbs get lemmatized correctly with POS tags
print(f'Original: He is walking to school and suffering from troubles')
print(f'Lemmatized: {lemmatize_sent("He is walking to school and suffering from troubles")}')

Original: He is walking to school and suffering from troubles
Lemmatized: ['he', 'be', 'walk', 'to', 'school', 'and', 'suffer', 'from', 'trouble']


In [19]:
# Apply lemmatization + stopword removal on the Hamlet passage
print('Original:')
print(sample_passage, '\n')

hamlet_clean = [word for word in lemmatize_sent(sample_passage)
                if word not in stoplist_combined and not word.isdigit()]

print('Lemmatized and cleaned:')
print(hamlet_clean)

Original:
To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles. 

Lemmatized and cleaned:
['question', "'t", 'nobler', 'mind', 'suffer', 'sling', 'arrow', 'outrageous', 'fortune', 'arm', 'sea', 'trouble']


In [20]:
def preprocess_text(text):
    """Full preprocessing: tokenize, lowercase, lemmatize, remove stopwords."""
    return [word for word in lemmatize_sent(text)
            if word not in stoplist_combined and not word.isdigit()]

# Test the full pipeline
print('Full pipeline result:')
print(preprocess_text(sample_passage))

Full pipeline result:
['question', "'t", 'nobler', 'mind', 'suffer', 'sling', 'arrow', 'outrageous', 'fortune', 'arm', 'sea', 'trouble']


---
## 7. Applying NLP Pipeline to Full Hamlet Text
Let's process a chunk of Hamlet and see the most common words.

In [21]:
# Process first 10000 characters of Hamlet
hamlet_sample_text = hamlet_raw[:10000]
hamlet_processed = preprocess_text(hamlet_sample_text)

print(f'Processed tokens: {len(hamlet_processed)}')
print(f'First 30 tokens: {hamlet_processed[:30]}')

Processed tokens: 860
First 30 tokens: ['tragedie', 'hamlet', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'centinels', 'barnardo', "'s", 'fran', 'nay', 'answer', 'stand', 'vnfold', 'selfe', 'bar', 'long', 'liue', 'king', 'fran', 'barnardo', 'bar', 'fran', 'carefully', 'vpon']


In [22]:
# Count word frequencies
hamlet_word_counts = Counter(hamlet_processed)

print('Top 20 most frequent words in Hamlet:')
for word, count in hamlet_word_counts.most_common(20):
    print(f'  {word}: {count}')

Top 20 most frequent words in Hamlet:
  mar: 17
  hor: 14
  'd: 14
  speake: 14
  haue: 13
  king: 12
  barn: 10
  night: 10
  horatio: 9
  's: 8
  fran: 8
  't: 8
  hath: 8
  thou: 8
  bar: 7
  againe: 7
  hamlet: 6
  enter: 6
  barnardo: 6
  watch: 6


---
## 8. From Strings to Vectors (Bag-of-Words)
In Bag-of-Words, each document becomes a vector of word counts. Each position represents a word, and the value is how many times it appears.

In [23]:
# Demonstrate BoW on two sentences
sent1 = "To be, or not to be, that is the question."
sent2 = "Whether tis nobler in the mind to suffer."

processed_sent1 = preprocess_text(sent1)
processed_sent2 = preprocess_text(sent2)

print('Sentence 1:', processed_sent1)
print('Counts:', Counter(processed_sent1))
print()
print('Sentence 2:', processed_sent2)
print('Counts:', Counter(processed_sent2))

Sentence 1: ['question']
Counts: Counter({'question': 1})

Sentence 2: ['ti', 'nobler', 'mind', 'suffer']
Counts: Counter({'ti': 1, 'nobler': 1, 'mind': 1, 'suffer': 1})


---
## 9. Vectorization with sklearn's CountVectorizer
CountVectorizer builds the vocabulary and converts text into count vectors. We plug in our `preprocess_text` function as the analyzer.

In [24]:
from io import StringIO
from sklearn.feature_extraction.text import CountVectorizer
from operator import itemgetter

sent1 = "To be, or not to be, that is the question."
sent2 = "Whether tis nobler in the mind to suffer."

# Create vectorizer with our custom preprocessing
with StringIO('\n'.join([sent1, sent2])) as fin:
    count_vect = CountVectorizer(analyzer=preprocess_text)
    count_vect.fit_transform(fin)

print('Vocabulary:', count_vect.vocabulary_)

Vocabulary: {'question': 2, 'ti': 4, 'nobler': 1, 'mind': 0, 'suffer': 3}


In [25]:
# Show the count vectors as a matrix
words_sorted, _ = zip(*sorted(count_vect.vocabulary_.items(), key=itemgetter(1)))

print('Sentence 1:', preprocess_text(sent1))
print('Sentence 2:', preprocess_text(sent2))
print(f'\nVocabulary: {words_sorted}')
print(f'\nCount Matrix:\n{count_vect.transform([sent1, sent2]).toarray()}')

Sentence 1: ['question']
Sentence 2: ['ti', 'nobler', 'mind', 'suffer']

Vocabulary: ('mind', 'nobler', 'question', 'suffer', 'ti')

Count Matrix:
[[0 0 1 0 0]
 [1 1 0 1 1]]


---
## 10. Naive Bayes Classification on Brown Corpus
Now we build a Naive Bayes classifier using the Brown Corpus. Our goal is to classify text as either **news** or **fiction**.

In [26]:
from nltk.corpus import brown

# Brown Corpus has text organized by genre
print('Available genres:')
print(brown.categories())

Available genres:
['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']


In [27]:
# Get file IDs for news and fiction genres
news_fileids = brown.fileids(categories='news')
fiction_fileids = brown.fileids(categories='fiction')

print(f'News files: {len(news_fileids)}')
print(f'Fiction files: {len(fiction_fileids)}')

News files: 44
Fiction files: 29


In [28]:
# Build dataset - each document is the text of one file, label is the genre
documents = []
labels = []

for fid in news_fileids:
    documents.append(' '.join(brown.words(fileids=fid)))
    labels.append('news')

for fid in fiction_fileids:
    documents.append(' '.join(brown.words(fileids=fid)))
    labels.append('fiction')

# Put into a DataFrame
df_brown = pd.DataFrame({'text': documents, 'genre': labels})

print(f'Total documents: {len(df_brown)}')
print(f'\nGenre counts:')
print(df_brown['genre'].value_counts())
print(f'\nSample (first 200 chars): {df_brown["text"].iloc[0][:200]}')

Total documents: 73

Genre counts:
genre
news       44
fiction    29
Name: count, dtype: int64

Sample (first 200 chars): The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place . The jury further said in term-end present


In [29]:
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% validation
train_data, valid_data = train_test_split(df_brown, test_size=0.2, random_state=42)

print(f'Training set: {len(train_data)} documents')
print(f'Validation set: {len(valid_data)} documents')

Training set: 58 documents
Validation set: 15 documents


In [30]:
from sklearn.feature_extraction.text import CountVectorizer

# Vectorize text using our preprocessing pipeline
count_vect = CountVectorizer(analyzer=preprocess_text)

# fit_transform on training data - builds vocabulary and creates vectors
train_vectors = count_vect.fit_transform(train_data['text'])
train_labels = train_data['genre']

# transform only on validation data - uses same vocabulary
valid_vectors = count_vect.transform(valid_data['text'])
valid_labels = valid_data['genre']

print(f'Training vectors: {train_vectors.shape}')
print(f'Validation vectors: {valid_vectors.shape}')
print(f'Vocabulary size: {len(count_vect.vocabulary_)}')

Training vectors: (58, 11371)
Validation vectors: (15, 11371)
Vocabulary size: 11371


In [31]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Train the Naive Bayes classifier
clf = MultinomialNB()
clf.fit(train_vectors, train_labels)

print('Naive Bayes classifier trained!')

Naive Bayes classifier trained!


In [32]:
# Test on validation set
predictions = clf.predict(valid_vectors)

accuracy = accuracy_score(valid_labels, predictions) * 100
print(f'Genre Classification Accuracy: {accuracy:.2f}%\n')

print('Classification Report:')
print(classification_report(valid_labels, predictions))

Genre Classification Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

     fiction       1.00      1.00      1.00         5
        news       1.00      1.00      1.00        10

    accuracy                           1.00        15
   macro avg       1.00      1.00      1.00        15
weighted avg       1.00      1.00      1.00        15



In [33]:
# Retrain on full dataset for best results
count_vect_full = CountVectorizer(analyzer=preprocess_text)
full_vectors = count_vect_full.fit_transform(df_brown['text'])
full_labels = df_brown['genre']

clf_final = MultinomialNB()
clf_final.fit(full_vectors, full_labels)

print(f'Final model trained on {len(df_brown)} documents')
print(f'Vocabulary size: {len(count_vect_full.vocabulary_)}')

Final model trained on 73 documents
Vocabulary size: 13177


In [34]:
# Test on new sentences the model has never seen
test_sentences = [
    "The president announced new economic policies at the press conference today.",
    "She walked through the dark forest, her heart pounding with every step.",
    "The stock market rallied after the Federal Reserve cut interest rates.",
    "He gazed into her eyes and whispered words she had longed to hear."
]

test_vectors = count_vect_full.transform(test_sentences)
test_predictions = clf_final.predict(test_vectors)

print('Predictions on New Sentences:')
print('=' * 70)
for sentence, prediction in zip(test_sentences, test_predictions):
    print(f'Text: "{sentence}"')
    print(f'Predicted Genre: {prediction}')
    print('-' * 70)

Predictions on New Sentences:
Text: "The president announced new economic policies at the press conference today."
Predicted Genre: news
----------------------------------------------------------------------
Text: "She walked through the dark forest, her heart pounding with every step."
Predicted Genre: fiction
----------------------------------------------------------------------
Text: "The stock market rallied after the Federal Reserve cut interest rates."
Predicted Genre: news
----------------------------------------------------------------------
Text: "He gazed into her eyes and whispered words she had longed to hear."
Predicted Genre: fiction
----------------------------------------------------------------------


---
## Conclusion

In this notebook, we performed a complete NLP pipeline on text from the Gutenberg Project (Shakespeare's Hamlet) and the Brown Corpus:

1. **Corpus Loading**: Loaded and explored Hamlet from NLTK's Gutenberg corpus.
2. **Tokenization**: Split text into sentences and words using NLTK's tokenizers.
3. **Lowercasing**: Converted all tokens to lowercase for consistency.
4. **Stopword Removal**: Removed common words using combined stopword lists.
5. **Stemming & Lemmatization**: Compared Porter Stemmer with WordNet Lemmatizer.
6. **Vectorization**: Converted text into numerical vectors using Bag-of-Words.
7. **Classification**: Built a Naive Bayes classifier to distinguish news from fiction, achieving strong accuracy.